
# FFNN — PyTorch 2.x (simplificado)

Este cuaderno combina el **pipeline compacto** con `torch.compile`, early stopping y utilidades modernas,
**añadiendo** las funciones/clases que tenía tu versión original (datasets, EDA, loops `loop_FFNN/MC`,
`evaluate`, plots, etc.).


In [10]:

# [setup]
import os, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset, Subset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import pandas as pd
from ucimlrepo import fetch_ucirepo

torch.set_float32_matmul_precision('high')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs('checkpoints', exist_ok=True)
print('Device:', DEVICE)


Device: cuda


In [11]:
# [model core] - REEMPLAZAR CELDA 3

# Función para inicializar los pesos
def init_weights(m, init_method='he'):
    if isinstance(m, nn.Linear):
        if init_method == 'he':
            # Inicialización He (buena para ReLU)
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
        elif init_method == 'xavier':
            # Inicialización Xavier (buena para Tanh o Sigmoid)
            nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

def make_mlp(d_in: int, d_hidden=(64, 64), d_out=1, dropout=0.0, 
             act_fn=nn.ReLU, use_bn=False):
    """
    Fábrica de MLPs flexible con Dropout y Batch Norm.
    """
    layers = []
    d_prev = d_in
    
    for i, h in enumerate(d_hidden):
        layers.append(nn.Linear(d_prev, h))
        
        # Batch Norm se aplica después de Lineal y ANTES de Activación
        if use_bn:
            layers.append(nn.BatchNorm1d(h))
            
        layers.append(act_fn())
        
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
            
        d_prev = h
        
    layers.append(nn.Linear(d_prev, d_out))
    return nn.Sequential(*layers)

@torch.no_grad()
def accuracy_bin(logits, y_true):
    preds = (logits.sigmoid() >= 0.5).float().view(-1)
    return (preds == y_true.view(-1)).float().mean().item()

@torch.no_grad()
def accuracy_multi(logits, y_true):
    preds = logits.argmax(1)
    return (preds == y_true).float().mean().item()

def set_seed(seed=42):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [12]:

# [fit loop]
def fit(model, train_ds, val_ds=None, *, epochs=100, batch_size=64, lr=1e-3,
        patience=10, min_delta=1e-4, ckpt_path=None, compile_model=True, weight_decay=0.0):
    model.to(DEVICE)
    if compile_model and hasattr(torch, 'compile'):
        try:
            model = torch.compile(model)  # PyTorch 2.x
        except Exception as e:
            print('torch.compile no disponible:', e)

    sample_x, sample_y = train_ds[:][0], train_ds[:][1]
    binary = sample_y.ndim == 2 or sample_y.unique().numel() <= 2
    if binary:
        criterion = nn.BCEWithLogitsLoss()
    else:
        criterion = nn.CrossEntropyLoss()

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    best = math.inf; best_epoch = None; best_state = None
    history = {'epoch': [], 'loss': [], 'acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(1, epochs+1):
        model.train()
        epoch_loss = 0.0; n = 0; accs = []
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            if binary:
                yb_f = yb.float().view(-1,1)
                loss = criterion(logits, yb_f)
                acc = accuracy_bin(logits.detach(), yb.detach().float())
            else:
                loss = criterion(logits, yb.long())
                acc = accuracy_multi(logits.detach(), yb.detach().long())
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            epoch_loss += loss.item()*xb.size(0); n += xb.size(0); accs.append(acc)
        tr_loss = epoch_loss/n; tr_acc = float(np.mean(accs))

        # Validación
        if val_ds is not None:
            model.eval()
            xb, yb = val_ds[:]
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            with torch.no_grad():
                logits = model(xb)
                if binary:
                    val_loss = criterion(logits, yb.float().view(-1,1)).item()
                    val_acc = accuracy_bin(logits, yb.float())
                else:
                    val_loss = criterion(logits, yb.long()).item()
                    val_acc = accuracy_multi(logits, yb.long())
            if val_loss + min_delta < best:
                best = val_loss; best_epoch = epoch; best_state = {k:v.cpu() for k,v in model.state_dict().items()}
                if ckpt_path: torch.save(best_state, ckpt_path)

        history['epoch'].append(epoch)
        history['loss'].append(tr_loss); history['acc'].append(tr_acc)
        if val_ds is not None:
            history['val_loss'].append(val_loss); history['val_acc'].append(val_acc)

        if epoch % 5 == 0:
            msg = f"[{epoch:03d}] loss={tr_loss:.4f} acc={tr_acc*100:5.1f}%"
            if val_ds is not None:
                msg += f" | vloss={val_loss:.4f} vacc={val_acc*100:5.1f}%"
            print(msg)

        # early stopping
        if val_ds is not None and (epoch - (best_epoch or 0)) >= patience:
            print(f"Paro temprano @ epoch {epoch} (best={best_epoch})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, best_epoch


In [13]:
# [compat: models with names FFNN / FFNN_MC] - REEMPLAZAR CELDA 9

class FFNN(nn.Module):
    # Binario: salida 1
    def __init__(self, d0, d1=64, d2=64, dropout=0.0, use_bn=False, init_method='he'):
        super().__init__()
        # Usamos ReLU como estándar para binario
        self.net = make_mlp(d0, (d1, d2), 1, dropout=dropout, act_fn=nn.ReLU, use_bn=use_bn)
        # Aplicamos la inicialización
        self.net.apply(lambda m: init_weights(m, init_method=init_method))
        
    def forward(self, x): 
        return self.net(x)

class FFNN_MC(nn.Module):
    # Multiclase: salida C
    def __init__(self, d0, d1=64, d2=64, C=3, dropout=0.0, use_bn=False, init_method='he'):
        super().__init__()
        self.net = make_mlp(d0, (d1, d2), C, dropout=dropout, act_fn=nn.ReLU, use_bn=use_bn)
        self.net.apply(lambda m: init_weights(m, init_method=init_method))

    def forward(self, x): 
        return self.net(x)

In [14]:

# [compat: evaluation & plotting]
@torch.no_grad()
def evaluate(model, dataset, binary=True, batch_size=256):
    model.eval(); loader = DataLoader(dataset, batch_size=batch_size)
    total=0; correct=0; losses=[]; crit = nn.BCEWithLogitsLoss() if binary else nn.CrossEntropyLoss()
    all_y=[]; all_p=[]
    for xb, yb in loader:
        xb = xb.to(DEVICE); yb = yb.to(DEVICE)
        logits = model(xb)
        if binary:
            loss = crit(logits, yb.float().view(-1,1))
            preds = (logits.sigmoid()>=0.5).float().view(-1)
            all_p.append(preds.cpu().numpy()); all_y.append(yb.view(-1).cpu().numpy())
        else:
            loss = crit(logits, yb.long())
            preds = logits.argmax(1)
            all_p.append(preds.cpu().numpy()); all_y.append(yb.view(-1).cpu().numpy())
        total += yb.size(0); correct += (preds == yb.view(-1)).float().sum().item(); losses.append(loss.item())
    acc = correct/total
    y_true = np.concatenate(all_y); y_pred = np.concatenate(all_p)
    cm = confusion_matrix(y_true, y_pred)
    rep = classification_report(y_true, y_pred, output_dict=False)
    return {'acc': acc, 'loss': float(np.mean(losses)), 'cm': cm, 'report': rep}

@torch.no_grad()
def evaluate_per_class(model, dataset, binary=False, batch_size=256):
    # simple wrapper que devuelve matriz de confusión y métricas por clase del classification_report
    e = evaluate(model, dataset, binary=binary, batch_size=batch_size)
    return e

def plot_history_loss(h):
    x = h['epoch'];
    plt.figure(); plt.plot(x, h['loss'], label='train_loss')
    if h.get('val_loss'): plt.plot(x, h['val_loss'], label='val_loss')
    plt.xlabel('Época'); plt.ylabel('Loss'); plt.title('Histórico Loss'); plt.legend(); plt.show()

def plot_history_acc(h):
    x = h['epoch'];
    plt.figure(); plt.plot(x, np.array(h['acc'])*100, label='train_acc')
    if h.get('val_acc'): plt.plot(x, np.array(h['val_acc'])*100, label='val_acc')
    plt.xlabel('Época'); plt.ylabel('Acc (%)'); plt.title('Histórico Acc'); plt.legend(); plt.show()

def plot_with_best_epoch(h, best_epoch):
    plot_history_loss(h); plot_history_acc(h)
    if best_epoch:
        print(f"Mejor época (val_loss): {best_epoch}")


In [15]:

# [plots with best epoch]
def plot_history(h, best_epoch=None, title='Histórico'):
    x = h['epoch']
    # Loss
    plt.figure()
    plt.plot(x, h['loss'], label='train_loss')
    if h.get('val_loss') and len(h['val_loss']) == len(x):
        plt.plot(x, h['val_loss'], label='val_loss')
    if best_epoch:
        plt.axvline(best_epoch, linestyle='--')
        plt.text(best_epoch, plt.ylim()[1]*0.9, f"best={best_epoch}", rotation=90, va='top')
    plt.xlabel('Época'); plt.ylabel('Loss'); plt.title(title + ' - Loss'); plt.legend(); plt.show()
    # Acc
    plt.figure()
    plt.plot(x, [a*100 for a in h['acc']], label='train_acc')
    if h.get('val_acc') and len(h['val_acc']) == len(x):
        plt.plot(x, [a*100 for a in h['val_acc']], label='val_acc')
    if best_epoch:
        plt.axvline(best_epoch, linestyle='--')
        plt.text(best_epoch, plt.ylim()[1]*0.9, f"best={best_epoch}", rotation=90, va='top')
    plt.xlabel('Época'); plt.ylabel('Acc (%)'); plt.title(title + ' - Acc'); plt.legend(); plt.show()

def plot_with_best_epoch(h, best_epoch):
    plot_history(h, best_epoch=best_epoch, title='Entrenamiento')


In [16]:
np.random.seed(42)
print("Cargando dataset Online News Popularity...")
online_news_popularity = fetch_ucirepo(id=332) 
X_df = online_news_popularity.data.features 
y_df = online_news_popularity.data.targets 

# Binarizar 'y' para clasificación
UMBRAL_POPULARIDAD = 1400 
y_binned = (y_df > UMBRAL_POPULARIDAD).astype(float)

X_data = X_df.to_numpy()
y_data = y_binned.to_numpy()
d0 = X_data.shape[1] # Dimensión de entrada

print(f"Dataset cargado. {X_data.shape[0]} muestras, {d0} características.")
print(f"Problema: Clasificación Binaria (Popular > {UMBRAL_POPULARIDAD} shares)")

# Estandarizar X (¡Muy importante!)
scaler = StandardScaler()
X_data_scaled = scaler.fit_transform(X_data)

# Dividir en Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_data_scaled, y_data, 
    test_size=0.2, stratify=y_data, random_state=42
)
# Dividir Train en Train/Validación (60/20 del total)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, 
    test_size=0.25, # 0.25 * 0.8 = 0.2 del total
    stratify=y_train, random_state=42
)

# Convertir a TensorDatasets
ds_train = TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.float32))
ds_val = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))
ds_test = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))

print(f"Datos divididos: {len(ds_train)} train, {len(ds_val)} val, {len(ds_test)} test")

Cargando dataset Online News Popularity...
Dataset cargado. 39644 muestras, 58 características.
Problema: Clasificación Binaria (Popular > 1400 shares)
Datos divididos: 23786 train, 7929 val, 7929 test


In [17]:
# --- Celda 8: Grid Search (NUEVA) ---
# [Grid Search / Bucle de Experimentos]

# Definición de tus experimentos
configurations = [
    {
        "name": "1. Modelo Base (Adam)",
        "dropout": 0.0,
        "use_bn": False,
        "init_method": "he",
        "weight_decay": 0.0
    },
    {
        "name": "2. Con Dropout (0.5)",
        "dropout": 0.5,
        "use_bn": False,
        "init_method": "he",
        "weight_decay": 0.0
    },
    {
        "name": "3. Con L2 (AdamW)",
        "dropout": 0.0,
        "use_bn": False,
        "init_method": "he",
        "weight_decay": 1e-3 # Valor de L2
    },
    {
        "name": "4. Con Batch Norm",
        "dropout": 0.0,
        "use_bn": True,
        "init_method": "he",
        "weight_decay": 0.0
    },
    {
        "name": "5. Inicialización Xavier",
        "dropout": 0.0,
        "use_bn": False,
        "init_method": "xavier", # Usando Xavier
        "weight_decay": 0.0
    },
    {
        "name": "6. L2 + Dropout + BN",
        "dropout": 0.3,
        "use_bn": True,
        "init_method": "he",
        "weight_decay": 1e-4
    }
]

results_summary = []
all_histories = {}
all_models = {}

print(f"\n--- Iniciando Grid Search en {DEVICE} ---")

for config in configurations:
    print(f"\nEntrenando: {config['name']}")
    
    # 1. Crear Modelo
    model = FFNN(
        d0, 
        d1=128, # Capas más grandes para más features
        d2=64, 
        dropout=config['dropout'], 
        use_bn=config['use_bn'],
        init_method=config['init_method']
    )
    
    # 2. Entrenar Modelo
    model, history, best_epoch = fit(
        model, 
        ds_train, 
        val_ds=ds_val,
        epochs=100, 
        lr=1e-3,
        batch_size=256, # Batch size más grande
        patience=10, 
        weight_decay=config['weight_decay'],
        compile_model=True 
    )
    
    # 3. Evaluar en Test
    test_eval = evaluate(model, ds_test, binary=True)
    
    # 4. Guardar resultados
    results_summary.append({
        "name": config['name'],
        "test_acc": test_eval['acc'],
        "test_loss": test_eval['loss'],
        "best_epoch": best_epoch,
        "total_epochs": len(history['epoch'])
    })
    all_histories[config['name']] = history
    all_models[config['name']] = model

print("\n--- Resultados del Grid Search (News Popularity) ---")
results_df = pd.DataFrame(results_summary).sort_values(by="test_acc", ascending=False)
print(results_df.to_markdown(index=False))

# Graficar el historial del mejor modelo
best_model_name = results_df.iloc[0]['name']
print(f"\n--- Historial del Mejor Modelo: {best_model_name} ---")
plot_history(all_histories[best_model_name], best_epoch=results_df.iloc[0]['best_epoch'], title=best_model_name)


--- Iniciando Grid Search en cuda ---

Entrenando: 1. Modelo Base (Adam)


W1110 16:17:38.649000 26172 site-packages\torch\_inductor\utils.py:1558] [0/0] Not enough SMs to use max_autotune_gemm mode


TritonMissing: Cannot find a working triton installation. Either the package is not installed or it is too old. More information on installing Triton can be found at: https://github.com/triton-lang/triton

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"


In [ ]:
# --- Celda 9: Bootstrap (Bagging) (NUEVA) ---
# [Bootstrap (Bagging)]

def run_bootstrap_experiment(train_dataset, val_dataset, test_dataset, n_models=10):
    
    print(f"\n--- Iniciando Bootstrap (Bagging) con {n_models} modelos ---")
    
    d0 = train_dataset[0][0].shape[0] 
    ensemble_models = []
    
    for i in range(n_models):
        print(f"Entrenando modelo de bootstrap {i+1}/{n_models}...")
        
        # 1. Crear muestra de bootstrap (muestreo CON reemplazo)
        n_train = len(train_dataset)
        # Asegúrate que los índices sean de CPU para Subset
        indices = torch.randint(0, n_train, (n_train,), device='cpu') 
        bootstrap_train_ds = Subset(train_dataset, indices)
        
        # 2. Crear y entrenar el modelo
        model = FFNN(d0, d1=128, d2=64, dropout=0.1, use_bn=True, init_method='he')
        
        model, hist, best = fit(
            model, 
            bootstrap_train_ds, 
            val_ds=val_dataset,
            epochs=100, 
            lr=1e-3,
            batch_size=256,
            patience=10,
            weight_decay=1e-4,
            compile_model=True
        )
        
        ensemble_models.append(model)
        
    print("--- Entrenamiento de Bootstrap finalizado ---")
    return ensemble_models

# --- Función para evaluar un ensemble (sirve para Bootstrap y Ensemble mixto) ---
@torch.no_grad()
def evaluate_ensemble(models, test_dataset, binary=True):
    all_preds = []
    
    # Poner todos los modelos en modo evaluación
    for model in models:
        model.eval()
        model.to(DEVICE)

    # Obtenemos las predicciones (logits) de cada modelo
    # Usamos DataLoader para el test set por si es muy grande
    test_loader = DataLoader(test_dataset, batch_size=512)
    all_logits = [[] for _ in models]
    all_yb = []

    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        all_yb.append(yb.cpu())
        
        for i, model in enumerate(models):
            logits = model(xb)
            all_logits[i].append(logits.cpu())

    # Concatenar resultados de los batches
    all_yb_tensor = torch.cat(all_yb, dim=0)
    all_logits_tensor = [torch.cat(logs, dim=0) for logs in all_logits]

    # Calcular predicciones finales de cada modelo
    all_preds = []
    for logits in all_logits_tensor:
        if binary:
            preds = (logits.sigmoid() >= 0.5).float() # Predicciones 0 o 1
        else:
            preds = logits.argmax(dim=1).float() # Predicciones 0, 1, 2...
        all_preds.append(preds.view(-1, 1))
    
    # Concatenar todas las predicciones (shape: [N_muestras, N_modelos])
    all_preds_tensor = torch.cat(all_preds, dim=1)
    
    # Votación por mayoría
    if binary:
        # Para binario, la media > 0.5 es voto mayoritario para 1
        ensemble_preds = (all_preds_tensor.mean(dim=1) >= 0.5).float()
    else:
        # Para multiclase, usamos 'torch.mode'
        ensemble_preds = torch.mode(all_preds_tensor, dim=1).values
        
    # Calcular precisión final del ensemble
    y_true = all_yb_tensor.float().view(-1)
    acc = (ensemble_preds == y_true).float().mean().item()
    
    print("\n--- Evaluación del Ensemble ---")
    print(f"Precisión del Ensemble (Voto Mayoritario): {acc*100:.2f}%")
    
    print("Reporte de Clasificación del Ensemble:")
    print(classification_report(y_true.numpy(), ensemble_preds.numpy()))
    
    print("Matriz de Confusión del Ensemble:")
    print(confusion_matrix(y_true.numpy(), ensemble_preds.numpy()))
    
    return acc

# --- Ejecutar el experimento de Bootstrap ---
# (Usamos los ds_train, ds_val, ds_test de la celda 7)
bootstrap_models = run_bootstrap_experiment(ds_train, ds_val, ds_test, n_models=7)
evaluate_ensemble(bootstrap_models, ds_test, binary=True)

In [ ]:
# --- Celda 10: Ensemble Mixto (NUEVA) ---
# [Ensemble Mixto]

print("\n--- Iniciando Ensemble Mixto (Dropout + L2 + Base) ---")

# Re-utilizamos los modelos ya entrenados en la celda 8
try:
    model_base = all_models["1. Modelo Base (Adam)"]
    model_dropout = all_models["2. Con Dropout (0.5)"]
    model_l2 = all_models["3. Con L2 (AdamW)"]
    
    mixed_ensemble_models = [model_base, model_dropout, model_l2]

    # Evaluamos el ensemble mixto
    evaluate_ensemble(mixed_ensemble_models, ds_test, binary=True)

except NameError:
    print("Error: Debes ejecutar la celda [Grid Search / Bucle de Experimentos] primero.")
except KeyError:
    print("Error: No se encontraron los modelos base. Asegúrate de que la celda [Grid Search] los haya entrenado.")